In [39]:
import json
import urllib.request
import pandas as pd

In [40]:
EIA_KEY = "GozRJWBzQ6p6nakSoYilJsgXHps1ajIvfaTi4VWf"

In [41]:
def eia_seds_gasoline_ca(api_key: str, start: int = 2010, end: int = 2024) -> pd.DataFrame:

    # ── Fix 1: separate seriesId (MSN code) from stateId ───────────────────
    # seriesId = the MSN code alone, stateId = the state — they are two distinct facets.
    # "CAMGACD" (concatenated) was wrong; the API splits them.
    #
    # ── Fix 2: use urllib.request instead of requests ──────────────────────
    # requests.get() calls PreparedRequest.prepare_url() → requote_uri(),
    # which percent-encodes [ ] even in a pre-built URL string.
    # urllib.request.urlopen() sends the URL byte-for-byte as given.

    url = (
        f"https://api.eia.gov/v2/seds/data/"
        f"?api_key={api_key}"
        f"&frequency=annual"
        f"&data[0]=value"             # EIA needs literal brackets; urllib preserves them
        f"&facets[seriesId][]=MGACD"  # MSN code only — NOT "CAMGACD"
        f"&facets[stateId][]=CA"      # state as its own facet
        f"&start={start}"
        f"&end={end}"
        f"&length=500"
        f"&sort[0][column]=period"
        f"&sort[0][direction]=asc"
    )

    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as resp:
        payload = json.loads(resp.read().decode("utf-8"))

    # ── Defensive checks ────────────────────────────────────────────────────
    if "response" not in payload:
        raise ValueError(f"Unexpected response shape: {list(payload.keys())}\n{payload}")

    records = payload["response"]["data"]
    total   = payload["response"]["total"]
    print(f"Records returned: {len(records)} of {total} total")

    if not records:
        # Print a sample row from the metadata to inspect available column names
        print("No data returned. Response metadata:", json.dumps(payload["response"], indent=2)[:600])
        return pd.DataFrame()

    df = pd.DataFrame(records)
    print("Columns from API:", df.columns.tolist())   # inspect once, then remove

    df["value"]  = pd.to_numeric(df["value"], errors="coerce")
    df["period"] = df["period"].astype(int)

    # ── Unit conversion: thousand barrels → gallons ─────────────────────────
    # MGACD is in thousand barrels; 1 barrel = 42 gallons
    df["gallons"] = df["value"] * 1_000 * 42

    return df[["period", "stateId", "seriesId", "seriesDescription", "value", "unit", "gallons"]]

In [43]:
eia_mg = eia_seds_gasoline_ca(EIA_KEY, start=2010, end=2024)
print(eia_mg.to_string())

Records returned: 15 of 15 total
Columns from API: ['period', 'seriesId', 'seriesDescription', 'stateId', 'stateDescription', 'value', 'unit']
    period stateId seriesId                                  seriesDescription  value                     unit    gallons
0     2010      CA    MGACD  Motor gasoline price in the transportation sector  24.76  Dollars per million Btu  1039920.0
1     2011      CA    MGACD  Motor gasoline price in the transportation sector  30.51  Dollars per million Btu  1281420.0
2     2012      CA    MGACD  Motor gasoline price in the transportation sector  32.24  Dollars per million Btu  1354080.0
3     2013      CA    MGACD  Motor gasoline price in the transportation sector  31.08  Dollars per million Btu  1305360.0
4     2014      CA    MGACD  Motor gasoline price in the transportation sector  29.99  Dollars per million Btu  1259580.0
5     2015      CA    MGACD  Motor gasoline price in the transportation sector  25.47  Dollars per million Btu  1069740.0
6  

In [45]:
# Save to CSV
eia_mg.to_csv("eia_seds_gasoline_ca.csv", index=False)